# Quantum quench: the QELM reservoir starts from the ground state of a pure field

Issue #26. Before each input is processed, the reservoir is prepared in the ground state of a pure field Hamiltonian

$$H_0 = -h \sum_i A_i, \qquad A = Z \text{ (default) or } X,$$

and is then quenched to the QELM Hamiltonian

$$H(x) = h(x) \sum_i Z_i + \sum_{i<j} J_{ij} X_i X_j, \qquad h(x) = h_0 + \kappa x .$$

For $h > 0$ the ground state of $H_0$ is the product state with every spin along $+A$:

| `quench_axis` | $H_0$ | Ground state | $E_0 / N$ | Gap |
| --- | --- | --- | --- | --- |
| `"z"` (default) | $-h\sum_i Z_i$ | $\vert 0\rangle^{\otimes N}$ | $-h$ | $2h$ |
| `"x"` | $-h\sum_i X_i$ | $\vert +\rangle^{\otimes N} = H^{\otimes N}\vert 0\rangle^{\otimes N}$ | $-h$ | $2h$ |

## Physical justification

Both states are product states: they carry **zero entanglement**, so every correlation and all the entanglement measured after the quench are built by the reservoir dynamics alone. $\vert +\rangle^{\otimes N}$ is in addition invariant under the global spin flip $P = \prod_i X_i$, a perfectly symmetric starting point. Starting quantum reservoir dynamics from a simple product state and letting a disordered spin Hamiltonian scramble it is the standard protocol of quantum reservoir computing, introduced by

> K. Fujii and K. Nakajima, *Harnessing Disordered-Ensemble Quantum Dynamics for Machine Learning*, Phys. Rev. Applied **8**, 024030 (2017), arXiv:1602.08159.

## How the states are written as an NQS

- $\vert +\rangle^{\otimes N}$ is built by applying a Hadamard gate $(X_i + Z_i)/\sqrt 2$ to each spin of $\vert 0\rangle^{\otimes N}$ (`hadamard_plus_state`), and written **exactly** as an RBM (`plus_state_rbm`): zero biases, and hidden unit $j$ connected only to spin $j$ with weight $w = 1$. Since $\cosh(w s_j) = \cosh(w)$ for $s_j = \pm 1$, all amplitudes are equal, while $\partial_{W_{kj}} \log\psi = s_k s_j \tanh w$ keeps every pair direction in the TDVP tangent space. The naive choice $W = 0$ is the same state, but its tangent space has no pair directions and TDVP freezes (negative control in the tests).
- $\vert 0\rangle^{\otimes N}$ is a single configuration, which an RBM can only approach with infinite biases (TDVP then freezes as well). The Z quench is therefore run in the **Hadamard frame** $U = \prod_i (X_i + Z_i)/\sqrt 2$: $U\vert 0\rangle^{\otimes N} = \vert +\rangle^{\otimes N}$, $U Z_i U = X_i$, $U X_i U = Z_i$. The features $\langle Z_i\rangle(t)$ from $\vert 0\rangle^{\otimes N}$ under $H(x)$ equal $\langle X_i\rangle(t)$ from $\vert +\rangle^{\otimes N}$ under $U H(x) U$. This is an exact change of basis; ED is always run in the lab frame, so the TDVP-vs-ED comparison checks it.

Unit checks: `tests/QELM/NQS/qelm/test_quench.py`.

In [ ]:
import matplotlib.pyplot as plt
import netket as nk
import numpy as np

from QELM.NQS.dynamics import (
    exact_time_evolution,
    expectation_values,
    run_tdvp,
    time_grid,
)
from QELM.NQS.embedding import VMCGroundStatePreparation, VMCSettings
from QELM.NQS.evaluation.metrics import NMSE
from QELM.NQS.measurements import pure_field_hamiltonian
from QELM.NQS.paths import experiment_paths
from QELM.NQS.qelm import (
    QELMReservoir,
    RidgeReadout,
    entanglement_entropy,
    feature_matrix,
    quench_ground_state,
    random_couplings,
)
from QELM.NQS.utils.outputs import find_repo_root
from QELM.NQS.variational import rbm, spin_half_hilbert, variational_state

N = 4
H0_FIELD = 1.0  # h of H_0: the VMC must find E/N = -h
QUENCH_AXIS = "z"  # "z" (default): H_0 = -h sum Z_i ; "x": H_0 = -h sum X_i
T = 1.0
SEED = 45

hilbert = spin_half_hilbert(N)
couplings = random_couplings(N, seed=7)
reservoir = QELMReservoir(
    couplings, h0=1.0, kappa=1.0, T=T, initial="quench", quench_axis=QUENCH_AXIS
)

paths = experiment_paths(
    experiment_name="quench_initialization", outputs_root=find_repo_root() / "outputs"
)
print("run folder:", paths.root)

## 1. Preparing the ground state of $H_0$ with VMC: $E/N \to -h$

VMC with stochastic reconfiguration (imaginary-time evolution projected on the RBM manifold) from a random RBM. The curve is the Monte Carlo estimate of $\langle H_0\rangle / N$ at each iteration, with its statistical error; the dashed line is the target $E_0/N = -h$. At the end, $\langle H_0\rangle / N$ is also summed exactly over the $2^N$ configurations, and the fidelity with the exact ground state is printed. The variational principle forbids any value below $-h$.

For the Z axis the target $\vert 0\rangle^{\otimes N}$ is only reached asymptotically (growing visible biases), which is why the dynamics below starts from the exact state instead of the VMC output.

In [ ]:
H0 = pure_field_hamiltonian(hilbert, h=H0_FIELD, axis=QUENCH_AXIS)
vmc_state = variational_state(hilbert, rbm(), seed=SEED, kind="mc", n_samples=2048)
prep = VMCGroundStatePreparation(
    H0,
    vmc_state,
    settings=VMCSettings(n_iter=300, n_samples=2048),
    paths=paths,
    metadata={"H0": f"-{H0_FIELD} sum {QUENCH_AXIS.upper()}_i", "seed": SEED},
)
prep.run()

exact_vmc = nk.vqs.FullSumState(hilbert, vmc_state.model, variables=vmc_state.variables)
e_per_spin = float(complex(exact_vmc.expect(H0).mean).real) / N
psi_vmc = np.array(exact_vmc.to_array())
psi_vmc = psi_vmc / np.linalg.norm(psi_vmc)
fidelity = abs(np.vdot(quench_ground_state(hilbert, QUENCH_AXIS), psi_vmc)) ** 2
print(f"exact <H_0>/N after VMC = {e_per_spin:.10f}   (target -h = {-H0_FIELD})")
print(
    f"|E/N + h| = {abs(e_per_spin + H0_FIELD):.2e}, "
    f"fidelity with the ground state = {fidelity:.8f}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(
    prep.iterations,
    prep.energy / N,
    yerr=prep.energy_error / N,
    marker=".",
    ms=3,
    lw=1,
    label=r"VMC $\langle H_0 \rangle / N$",
)
ax.axhline(
    -H0_FIELD, color="k", linestyle="--", label=rf"target $E_0/N = -h = {-H0_FIELD:g}$"
)
ax.set_xlabel("iteration")
ax.set_ylabel(r"$E / N$")
ax.set_title(rf"VMC on $H_0 = -h\sum_i {QUENCH_AXIS.upper()}_i$, N = {N}")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.savefig(paths.plots / "vmc_energy_per_spin.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Quench dynamics from the exact ground state

One input $x = 0.3$. ED (lab frame) and TDVP (exact RBM, Hadamard frame for the Z quench) for the features $\langle Z_i\rangle(t)$, the energy $\langle H(x)\rangle(t)/N$, which a quench conserves exactly (dashed: its $t=0$ value, $h(x)$ for $\vert 0\rangle^{\otimes N}$ and $\sum_{i<j}J_{ij}/N$ for $\vert +\rangle^{\otimes N}$), and the half-system entanglement entropy built from zero by the reservoir.

In [ ]:
x = 0.3
grid = time_grid(T, reservoir.tdvp.dt)
H = reservoir.hamiltonian(x)
states = exact_time_evolution(H, reservoir.initial_vector(), grid)
z_ed = np.stack(
    [expectation_values(states, op) for op in reservoir.observables().values()], axis=1
)
e_ed = expectation_values(states, H)
entropy = np.array([entanglement_entropy(s, N, range(N // 2)) for s in states])

H_tdvp = reservoir.hamiltonian(x, frame="tdvp")
obs = {**reservoir.observables(frame="tdvp"), "E": H_tdvp}
result = run_tdvp(
    H_tdvp, reservoir.initial_state(), T=T, observables=obs, settings=reservoir.tdvp
)
z_tdvp = np.stack([result.observables[f"Z{i}"] for i in range(N)], axis=1)
e_tdvp = result.observables["E"]

print(f"max |<Z_i>_TDVP - <Z_i>_ED| = {np.abs(z_tdvp - z_ed).max():.2e}")
print(
    f"energy drift: ED {np.abs(e_ed - e_ed[0]).max():.2e}, "
    f"TDVP {np.abs(e_tdvp - e_ed[0]).max():.2e}"
)
print(
    f"S(t=0) = {entropy[0]:.2e},  S(T) = {entropy[-1]:.4f}  "
    f"(max {N // 2} ln 2 = {N // 2 * np.log(2):.4f})"
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(N):
    axes[0].plot(grid, z_ed[:, i], color=f"C{i}", label=f"ED $Z_{i}$")
    axes[0].plot(result.times, z_tdvp[:, i], color=f"C{i}", linestyle=":", lw=2)
axes[0].set(
    xlabel="t", ylabel=r"$\langle Z_i \rangle$", title="features (dotted: TDVP)"
)
axes[1].plot(grid, e_ed / N, label="ED")
axes[1].plot(result.times, e_tdvp / N, linestyle=":", lw=2, label="TDVP")
axes[1].axhline(e_ed[0] / N, color="k", linestyle="--", label="conserved value")
axes[1].set(xlabel="t", ylabel=r"$\langle H(x) \rangle / N$", title="energy per spin")
# Linear axis around the conserved value: the drift (~1e-10) is invisible.
axes[1].set_ylim(e_ed[0] / N - 0.1, e_ed[0] / N + 0.1)
axes[1].ticklabel_format(useOffset=False)
axes[2].plot(grid, entropy)
axes[2].set(
    xlabel="t",
    ylabel=r"$S_{A}$ (nats)",
    title=f"entanglement, A = first {N // 2} spins",
)
for ax in axes:
    ax.grid(True, linestyle="--", alpha=0.6)
axes[0].legend(fontsize=7)
axes[1].legend()
fig.tight_layout()
fig.savefig(paths.plots / "quench_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Baseline QELM metrics with each initial state

Ridge readout ($\lambda = 10^{-6}$) on 11 training inputs, NMSE on 10 held-out inputs in $[-1, 1]$, for the random RBM of the previous experiments and the two quenches. TDVP takes a few seconds per input.

In [ ]:
xs = np.linspace(-1.0, 1.0, 21)
train, test = xs[::2], xs[1::2]
targets = {"x^2": xs**2, "sin(pi x)": np.sin(np.pi * xs)}
setups = {
    "random RBM": QELMReservoir(couplings, T=T),
    "quench Z": QELMReservoir(couplings, T=T, initial="quench", quench_axis="z"),
    "quench X": QELMReservoir(couplings, T=T, initial="quench", quench_axis="x"),
}
print(
    f"{'initial state':<12} {'method':<6} "
    + " ".join(f"{name:>12}" for name in targets)
)
for label, r in setups.items():
    for method in ("exact", "tdvp"):
        F_train = feature_matrix(r, train, method=method)
        F_test = feature_matrix(r, test, method=method)
        scores = []
        for y in targets.values():
            readout = RidgeReadout(lam=1e-6).fit(F_train, y[::2])
            scores.append(NMSE(y[1::2], readout.predict(F_test)))
        print(f"{label:<12} {method:<6} " + " ".join(f"{s:12.2e}" for s in scores))